**Manual Single-Neuron Pass**
- **Task:** Define a single neuron with 3 inputs: w=[0.2, 0.4, 0.1], b=0.5, and input x=[1.0, 2.0, 3.0]. Using purely pen/paper or simple Python (no autograd), calculate the forward pass (using Sigmoid). Then, assuming a target y=1 and MSE loss, calculate the exact gradient of the loss with respect to each weight and the bias.
- **Why:** Solidifies the calculus (Chain Rule) behind backpropagation without framework abstractions hiding the math.

In [1]:
import math

# Inputs
x = [1.0, 2.0, 3.0]
w = [0.2, 0.4, 0.1]
b = 0.5
y = 1.0

# Forward Pass
z = sum(wi * xi for wi, xi in zip(w, x)) + b
y_hat = 1 / (1 + math.exp(-z))
loss = 0.5 * (y - y_hat) ** 2

# Backward Pass
dL_dyhat = y_hat - y
dyhat_dz = y_hat * (1 - y_hat)
dL_dz = dL_dyhat * dyhat_dz

dL_dw = [dL_dz * xi for xi in x]
dL_db = dL_dz * 1

# Output Results
print(f"--- Forward Pass ---")
print(f"z       = {z:.7f}")
print(f"y_hat   = {y_hat:.7f}")
print(f"Loss    = {loss:.7f}\n")

print(f"--- Backward Pass ---")
print(f"dL/dw1  = {dL_dw[0]:.7f}")
print(f"dL/dw2  = {dL_dw[1]:.7f}")
print(f"dL/dw3  = {dL_dw[2]:.7f}")
print(f"dL/db   = {dL_db:.7f}")

--- Forward Pass ---
z       = 1.8000000
y_hat   = 0.8581489
Loss    = 0.0100609

--- Backward Pass ---
dL/dw1  = -0.0172674
dL/dw2  = -0.0345349
dL/dw3  = -0.0518023
dL/db   = -0.0172674


**Build "Micrograd" (Scalar Autograd)**
- **Task:** Following Andrej Karpathy's "Micrograd" concept, build a Python class Value that stores a scalar number and its gradient. Implement dunder methods (__add__, __mul__) so that when you perform math, it automatically builds a directed acyclic graph (DAG) and knows how to compute the local gradient for backprop.
- **Why:** The ultimate exercise to understand how dynamic computation graphs (like PyTorch's) actually work.

In [2]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            print(f"[{self._op if self._op else 'Leaf'}] Adding out.grad ({out.grad}) to self.grad ({self.grad}) -> {self.grad + out.grad}")
            self.grad += out.grad
            print(f"[{other._op if other._op else 'Leaf'}] Adding out.grad ({out.grad}) to other.grad ({other.grad}) -> {other.grad + out.grad}")
            other.grad += out.grad

        out._backward = _backward
        print(f"[Forward +] {self.data} + {other.data} = {out.data}")
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            local_self_grad = other.data * out.grad
            local_other_grad = self.data * out.grad
            print(f"[{self._op if self._op else 'Leaf'}] Adding local grad ({local_self_grad}) to self.grad ({self.grad}) -> {self.grad + local_self_grad}")
            self.grad += local_self_grad
            print(f"[{other._op if other._op else 'Leaf'}] Adding local grad ({local_other_grad}) to other.grad ({other.grad}) -> {other.grad + local_other_grad}")
            other.grad += local_other_grad

        out._backward = _backward
        print(f"[Forward *] {self.data} * {other.data} = {out.data}")
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        print("\n--- Starting Backward Pass (Topological Order) ---")
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


# --- Verification Execution ---
print("--- Starting Forward Pass ---")
a = Value(2.0)
b = Value(4.0)
c = Value(3.0)

# (a * b) + c
d = a * b
e = d + c

e.backward()

print("\n--- Final Gradients ---")
print(f"a: {a}")
print(f"b: {b}")
print(f"c: {c}")
print(f"d: {d}")
print(f"e: {e}")

--- Starting Forward Pass ---
[Forward *] 2.0 * 4.0 = 8.0
[Forward +] 8.0 + 3.0 = 11.0

--- Starting Backward Pass (Topological Order) ---
[*] Adding out.grad (1.0) to self.grad (0.0) -> 1.0
[Leaf] Adding out.grad (1.0) to other.grad (0.0) -> 1.0
[Leaf] Adding local grad (4.0) to self.grad (0.0) -> 4.0
[Leaf] Adding local grad (2.0) to other.grad (0.0) -> 2.0

--- Final Gradients ---
a: Value(data=2.0, grad=4.0)
b: Value(data=4.0, grad=2.0)
c: Value(data=3.0, grad=1.0)
d: Value(data=8.0, grad=1.0)
e: Value(data=11.0, grad=1.0)
